# 智能助手（Assistants）

[智能助手](https://langchain-ai.github.io/langgraph/concepts/assistants/#resources)为开发者提供了一种快速简便的方式来修改和版本化智能代理，以便进行实验和测试。

## 为图（Graph）提供配置

我们的 `task_maistro` 图已经配置好使用智能助手功能！

它包含一个 `configuration.py` 文件，该文件已在图中定义并加载。

我们可以在图的节点内部访问可配置的字段（`user_id`、`todo_category`、`task_maistro_role`）。

## 创建智能助手

现在，对于我们一直在构建的 `task_maistro` 应用程序，智能助手的实际用例是什么？

对我来说，就是能够为不同类别的任务维护独立的待办事项列表。

例如，我想要一个助手来管理我的个人任务，另一个助手来管理工作任务。

这些可以通过 `todo_category` 和 `task_maistro_role` 可配置字段轻松配置。

![智能助手配置界面截图](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [ ]:
# 安装和升级 LangGraph SDK
# 使用 %%capture 魔法命令来抑制安装过程中的输出信息
# --no-stderr 参数确保错误信息仍然会显示
%%capture --no-stderr
%pip install -U langgraph_sdk

这是我们部署图时创建的默认智能助手。

In [ ]:
# 导入 LangGraph SDK 客户端
from langgraph_sdk import get_client

# 设置本地部署的 URL 地址
# 这是通过 CLI 部署的本地服务地址
url_for_cli_deployment = "http://localhost:8123"

# 创建客户端实例，用于与部署的图进行交互
client = get_client(url=url_for_cli_deployment)

### 个人助手

这是我用来管理个人任务的个人智能助手。

In [ ]:
# 创建个人助手
# 使用客户端创建一个新的智能助手实例
personal_assistant = await client.assistants.create(
    # "task_maistro" 是我们部署的图的名称
    "task_maistro", 
    # 配置参数：设置待办事项类别为"personal"（个人）
    config={"configurable": {"todo_category": "personal"}}
)

# 打印创建的助手信息，包括助手ID、配置等详细信息
print(personal_assistant)

{'assistant_id': 'e6ab9c39-4b56-4db9-bb39-a71484c5d408', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.897312+00:00', 'updated_at': '2025-07-31T18:33:39.897312+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'metadata': {}, 'version': 1, 'name': 'Untitled', 'description': None, 'context': {}}


让我们更新这个助手，为了方便起见添加我的 `user_id`，[创建它的新版本](https://langchain-ai.github.io/langgraph/cloud/how-tos/assistant_versioning/#create-a-new-version-for-your-assistant)。 

In [ ]:
# 定义个人助手的角色和职责
# 这是一个详细的系统提示词，定义了助手的行为模式和沟通风格
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

# 配置参数组合
# 包含待办事项类别、用户ID和助手角色定义
configurations = {"todo_category": "personal", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

# 更新个人助手配置
# 使用助手ID和新的配置参数来更新现有助手
personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)

# 打印更新后的助手信息
print(personal_assistant)

{'assistant_id': 'e6ab9c39-4b56-4db9-bb39-a71484c5d408', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.908742+00:00', 'updated_at': '2025-07-31T18:33:39.908742+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'personal', 'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging

### 工作助手

现在，让我们创建一个工作助手。我将用它来管理工作任务。

In [ ]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {"todo_category": "work", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

# 创建工作助手
# 使用客户端创建一个新的工作助手实例
work_assistant = await client.assistants.create(
    # "task_maistro" 是我们部署的图的名称
    "task_maistro", 
    # 配置参数：设置待办事项类别为"work"（工作）
    config={"configurable": configurations}
)

# 打印创建的工作助手信息
print(work_assistant)

{'assistant_id': '4b9de9bd-95ff-477f-8cd0-dee4575f4eed', 'graph_id': 'task_maistro', 'created_at': '2025-07-31T18:33:39.914775+00:00', 'updated_at': '2025-07-31T18:33:39.914775+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'work', 'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 day

## 使用智能助手

智能助手将保存到我们部署中的 `Postgres` 数据库中。

这使我们能够轻松地使用 SDK [搜索](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)智能助手。

In [ ]:
# 搜索所有智能助手
# 使用客户端搜索功能获取所有已创建的助手
assistants = await client.assistants.search()

# 遍历并打印每个助手的基本信息
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],  # 助手的唯一标识符
        'version': assistant['version'],            # 助手的版本号
        'config': assistant['config']               # 助手的配置信息
    })

{'assistant_id': '4b9de9bd-95ff-477f-8cd0-dee4575f4eed', 'version': 1, 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'work', 'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks based on deadlines and team dependencies\n- Maintain a professional tone while helping 

我们可以使用 SDK 轻松管理这些助手。例如，我们可以删除不再使用的助手。
> 视频中的语法略有不同。下面更新的代码创建了一个备用助手，然后删除它。 

In [ ]:
# 创建一个临时的智能助手
# 用于演示删除功能
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

# 搜索所有助手并显示删除前的状态
assistants = await client.assistants.search()
for assistant in assistants:
    print(f"删除前: {{'assistant_id': {assistant['assistant_id']}}}")
    
# 删除我们的临时助手
# 删除最后一个助手（刚创建的临时助手）
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

# 再次搜索所有助手并显示删除后的状态
assistants = await client.assistants.search()
for assistant in assistants:
    print(f"删除后: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': f79e12f9-67f2-46c2-9b5b-e7fa6ad31355}
before delete: {'assistant_id': 4b9de9bd-95ff-477f-8cd0-dee4575f4eed}
before delete: {'assistant_id': e6ab9c39-4b56-4db9-bb39-a71484c5d408}
before delete: {'assistant_id': 4a2980c5-2812-4d8e-ae62-3fb72f9ef98f}
before delete: {'assistant_id': 4955437e-b617-4a25-8470-11f49f71f388}

after delete: {'assistant_id': f79e12f9-67f2-46c2-9b5b-e7fa6ad31355 }
after delete: {'assistant_id': 4b9de9bd-95ff-477f-8cd0-dee4575f4eed }
after delete: {'assistant_id': e6ab9c39-4b56-4db9-bb39-a71484c5d408 }
after delete: {'assistant_id': 4a2980c5-2812-4d8e-ae62-3fb72f9ef98f }


让我们设置我将要使用的 `personal`（个人）和 `work`（工作）助手的ID。

In [ ]:
# 设置助手ID变量，方便后续使用
# 从搜索结果中获取工作助手和个人助手的ID
work_assistant_id = assistants[0]['assistant_id']      # 工作助手的ID
personal_assistant_id = assistants[1]['assistant_id']  # 个人助手的ID

### 工作助手

让我们为我的工作助手添加一些待办事项。

In [ ]:
# 导入必要的消息类型和转换函数
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

# 用户输入：创建或更新待办事项
user_input = "Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday."

# 创建一个新的对话线程
# 线程用于管理用户与助手之间的对话历史
thread = await client.threads.create()

# 使用工作助手处理用户输入
# 通过流式处理方式获取助手的响应
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    # 当接收到值事件时，处理并显示助手的响应
    if chunk.event == 'values':
        state = chunk.data
        # 将消息转换为可读格式并打印最后一条消息
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create or update few ToDos: 1) Re-film Module 6, lesson 5 by end of day today. 2) Update audioUX by next Monday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_HLhZN3g4O7wnsyUH40j4Jhy7)
 Call ID: call_HLhZN3g4O7wnsyUH40j4Jhy7
  Args:
    update_type: todo
================================= Tool Message =================================

Document fc43950f-d854-4621-ba9d-c5ada1a74a7b unchanged:
The task 'Re-film Module 6, lesson 5' has a deadline of '2025-07-30T23:59:00', which is already set to the end of day today. No changes are needed for this task.

Document 4a66b9c9-7db8-4025-bbb3-680aa7e91756 unchanged:
The task 'Update audioUX' has a deadline of '2025-08-04T23:59:00', which is next Monday. No changes are needed for this task.
================================== Ai Message ==================================

I've updated your ToDo list 

In [ ]:
# 为工作助手新增一条待办
user_input = "Create another ToDo: Finalize set of report generation tutorials."

# 新建对话线程，用于承载对话上下文
thread = await client.threads.create()

# 流式运行：将用户消息发送给工作助手，并逐步读取返回值
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    # 当事件类型为 values 时，代表有新的状态返回
    if chunk.event == 'values':
        state = chunk.data
        # 打印助手的最后一条消息
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create another ToDo: Finalize set of report generation tutorials.
================================== Ai Message ==================================

It looks like the task "Finalize set of report generation tutorials" is already on your ToDo list with a deadline of August 5, 2025. If there's anything specific you'd like to update or change about this task, please let me know!


助手会根据其系统指令在创建任务时进行反馈和约束。

它会提示我补充任务的截止日期（deadline）。

In [ ]:
# 回复助手：为该任务设置下周二为截止日期
user_input = "OK, for this task let's get it done by next Tuesday."

# 继续在同一个线程中进行流式交互
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    # 处理返回的状态值
    if chunk.event == 'values':
        state = chunk.data
        # 打印助手的最后一条消息
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

OK, for this task let's get it done by next Tuesday.
================================== Ai Message ==================================

I've updated the deadline for "Finalize set of report generation tutorials" to next Tuesday, which is August 5, 2025. If there's anything else you'd like to adjust, feel free to let me know!


### 个人助手

同样地，我们也可以为个人助手添加待办事项。

In [ ]:
# 为个人助手批量创建两条待办
user_input = "Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points."

# 新建线程并以流式方式与个人助手交互
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    # 处理返回值事件
    if chunk.event == 'values':
        state = chunk.data
        # 打印助手的最后一条消息
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Create ToDos: 1) Check on swim lessons for the baby this weekend. 2) For winter travel, check AmEx points.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_SMG3ByOuLfbpE4AiulNNaaj9)
 Call ID: call_SMG3ByOuLfbpE4AiulNNaaj9
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Check on swim lessons for the baby this weekend', 'time_to_complete': 30}

New ToDo created:
Content: {'task': 'For winter travel, check AmEx points', 'time_to_complete': 45}
================================== Ai Message ==================================

I've added the tasks to your ToDo list:

1. Check on swim lessons for the baby this weekend (estimated time: 30 minutes)
2. For winter travel, check AmEx points (estimated time: 45 minutes)

I notice these tasks don't have de

In [ ]:
# 让个人助手给出当前待办摘要
user_input = "Give me a todo summary."

# 新建线程并请求摘要
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    # 处理流式返回并打印最终消息
    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Give me a todo summary.
================================== Ai Message ==================================

Here's your current ToDo summary:

**Overdue:**
- Re-film Module 6, lesson 5 (Deadline: 2025-07-30)

**Due This Week:**
- Update audioUX (Deadline: 2025-08-04)
- Finalize set of report generation tutorials (Deadline: 2025-08-05)

**No Deadline:**
- For winter travel, check AmEx points
- Check on swim lessons for the baby this weekend

**Notes:**
- The task "Update audioUX" doesn't have a time estimate. It might be important to add one to better manage your time.
- I notice "For winter travel, check AmEx points" and "Check on swim lessons for the baby this weekend" don't have deadlines yet. Would you like to set deadlines for these tasks?
